# Notebook 02 — Sliding-Window Surface Inspection Pipeline

**Module:** `vision/surface_inspection.py`  
**Dataset:** MVTec AD (any high-resolution category)  
**Model:** SurfaceInspector wrapping a trained DefectDetector

---

## Problem Statement

In industrial inline inspection, cameras capture high-resolution images of
entire product surfaces (e.g. 4096×4096 steel sheets). Running a classifier
on the full image is slow and loses spatial resolution.

The sliding-window pipeline:
1. Divides the surface into overlapping 224×224 patches
2. Scores each patch with the trained CNN
3. Aggregates scores into a spatial heatmap
4. Thresholds the heatmap to produce a binary defect mask

This notebook demonstrates the full pipeline end-to-end.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from vision.defect_detection import DefectDetector
from vision.surface_inspection import SurfaceInspector
from datasets.mvtec_loader import MVTecDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load a Pre-Trained Model

If you have a checkpoint from Notebook 01, load it here.
Otherwise we create a fresh (random weights) model for pipeline demonstration.

In [ ]:
CHECKPOINT_PATH = '../checkpoints/notebook01_resnet50.pth'
import os

model = DefectDetector(backbone='resnet50', num_classes=2, pretrained=True)

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
else:
    print('No checkpoint found. Using pretrained ImageNet weights (demo mode).')

model = model.to(DEVICE).eval()

## 2. Build the Inspector

In [ ]:
inspector = SurfaceInspector(
    model=model,
    patch_size=112,      # smaller patches → finer heatmap
    stride=56,           # 50% overlap
    defect_class=1,
    threshold=0.5,
    device=DEVICE,
)
print('Inspector ready.')
print(f'  Patch size: {inspector.patch_size}px')
print(f'  Stride:     {inspector.stride}px')
print(f'  Threshold:  {inspector.threshold}')

## 3. Run Inspection on Test Images

In [ ]:
MVTEC_ROOT = '../datasets/raw/mvtec_ad'
CATEGORY   = 'bottle'

import torchvision.transforms as T

# Load raw images WITHOUT normalization for the inspector
raw_transform = T.Compose([
    T.Resize((512, 512)),   # resize to a larger resolution to show sliding window
])

try:
    # Load raw PIL images directly
    from datasets.mvtec_loader import MVTecDataset
    from pathlib import Path
    test_dir = Path(MVTEC_ROOT) / CATEGORY / 'test'
    test_images = []
    test_labels = []
    for d in sorted(test_dir.iterdir()):
        if not d.is_dir(): continue
        label = 0 if d.name == 'good' else 1
        for img_path in sorted(d.glob('*.png'))[:3]:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((512, 512))
            test_images.append(img)
            test_labels.append(label)
    print(f'Loaded {len(test_images)} test images.')
except Exception as e:
    # Demo: create synthetic test images
    print(f'Dataset not found ({e}). Creating synthetic test images.')
    rng = np.random.default_rng(0)
    test_images = [Image.fromarray((rng.random((512, 512, 3)) * 255).astype(np.uint8)) for _ in range(4)]
    test_labels = [0, 1, 0, 1]

In [ ]:
n_show = min(4, len(test_images))
fig, axes = plt.subplots(n_show, 3, figsize=(14, 4 * n_show))

if n_show == 1:
    axes = axes[np.newaxis, :]

for i in range(n_show):
    img   = test_images[i]
    label = test_labels[i]
    
    result = inspector.inspect(img, return_heatmap=True)
    overlay = inspector.visualize(img, result, alpha=0.45)
    
    # Original
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Original ({'DEFECT' if label else 'GOOD'})',
                          color='red' if label else 'green', fontweight='bold')
    axes[i, 0].axis('off')
    
    # Heatmap
    axes[i, 1].imshow(result['heatmap'], cmap='hot', vmin=0, vmax=1)
    axes[i, 1].set_title(f'Defect Heatmap (defect_ratio={result["defect_ratio"]:.2%})')
    axes[i, 1].axis('off')
    
    # Overlay
    axes[i, 2].imshow(overlay)
    verdict = 'DEFECTIVE' if result['is_defective'] else 'CLEAN'
    axes[i, 2].set_title(f'Overlay — Verdict: {verdict}',
                          color='red' if result['is_defective'] else 'green')
    axes[i, 2].axis('off')

plt.suptitle('Sliding-Window Surface Inspection', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Effect of Stride on Heatmap Resolution

In [ ]:
img = test_images[1]  # take a defective sample
strides = [112, 56, 28]

fig, axes = plt.subplots(1, len(strides) + 1, figsize=(5 * (len(strides) + 1), 5))

axes[0].imshow(img)
axes[0].set_title('Original')
axes[0].axis('off')

for j, stride in enumerate(strides):
    insp = SurfaceInspector(model=model, patch_size=112, stride=stride, device=DEVICE)
    res = insp.inspect(img)
    n_patches = len(res['patch_results'])
    axes[j + 1].imshow(res['heatmap'], cmap='hot', vmin=0, vmax=1)
    axes[j + 1].set_title(f'Stride={stride}px\n({n_patches} patches)')
    axes[j + 1].axis('off')

plt.suptitle('Heatmap Resolution vs Stride', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

The sliding-window inspector:
- **Smaller stride** → finer heatmap, more patches, slower
- **Larger stride** → coarser heatmap, fewer patches, faster

For real-time inspection, tune stride to meet the line speed requirement.

**Next:** `03_robot_anomaly_detection.ipynb` — detecting anomalies in robot joint sensors.